<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 附录 E：批处理与面向吞吐量的执行

本 Notebook 使用的 Python 包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- 在正文章节中，我们通常一次处理一个样例
- 这样代码更紧凑，也更容易理解
- 但代码的运行成本本来就很高，受硬件和资源限制，加入批处理支持带来的收益也很有限
- 不过，在某些场景下，能够以批处理模式运行代码仍然很有用
- 本附录解释批处理执行背后的总体思路，并展示如何使用补充材料中的代码，将它应用到各个章节

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 批处理为何有帮助

- 有两个不同的性能目标：
  - 延迟：获得单个提示词答案的速度；
  - 吞吐量：单位时间内可以处理多少个提示词。
- 单样例生成通常最适合降低延迟和调试代码
- 批处理主要针对吞吐量
- 如果要评估 MATH-500 中的数百道题、生成许多自洽性样例，或在许多监督样例上训练，批处理可以在合适的硬件上大幅缩短总运行时间
  - 当然，批处理并不保证在每种设备上都更快
  - CPU 或某些优化不足的 GPU 上的小模型可能无法从批处理中受益；甚至可能变慢，因为额外的填充和批处理开销会抵消并行带来的收益

&nbsp;
## E.2 运行批量生成

- 批处理的主要技术障碍是：提示词通常长度不同
- 例如，一道数学题分词后可能是 40 个 token，另一道可能是 120 个 token
- 由于 PyTorch 中的张量必须具有矩形形状，我们要对较短的序列进行填充，让它们都能放入同一个批量张量

- 从概念上说，这使批量生成比单提示词生成难实现得多
- 正文章节中，我们使用 `reasoning_from_scratch.qwen3` 的 `Qwen3Model` 类（其 Qwen3 实现已在附录 C 中解释）
- 批量生成需要跟踪填充 token 等信息，因此 `reasoning_from_scratch.qwen3_batched` 中提供了一个单独的 `Qwen3Model` 类（源代码可在补充材料中查看：https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py）

- 为了说明批量生成工具的用法，先看一个具体例子
- 从正文章节使用过的单序列文本生成示例开始
- 这里把两个提示词（`["2+2?", "3+3=6?"]`）依次处理：

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 下面使用 `reasoning_from_scratch.qwen3_batched` 中支持批处理的类似代码
- 注意，批量版本不支持流式输出，也就是说必须等所有结果生成后，才能解码并打印
- 这里的批量生成使用左填充，下一节会解释这一点
- 现在先从用法示例开始，看看如何使用它，然后再介绍内部原理

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 可以看到，结果与前面完全相同
- 区别在于，这些结果是通过 `generate_text_basic_batched_cache` 并行生成的
- 下一节会简要解释其底层工作方式

- 还有一种更优化的代码实现，用 `generate_text_basic_batched_cache_stop` 替换 `generate_text_basic_batched_cache`
- `generate_text_basic_batched_cache` 在每个解码步骤中都会保留活动批次中的每一行
- `generate_text_basic_batched_cache_stop` 会从活动计算批次中移除已完成的行（内部实现更复杂，但可以优化性能）
- 下图展示了这一点

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- 旁注：在 Qwen3 中，`<eos>` token 实际上是 `<|endoftext|>`，但图中为了视觉简洁使用了 `<eos>`

&nbsp;
## E.3 填充与注意力掩码

- 在单样例模式下，如果把短提示词（如 `"2+2?"`）分词，可以将它作为形状为 `(1, 4)` 的简单张量传给模型：
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- 模型内部会构建标准的因果注意力掩码，使每个位置只能关注自身和之前的 token
- 如果你不熟悉自注意力，我写过一篇文章介绍更多背景：https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- 从概念上看，该掩码如下：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` 表示“被掩码”，`0` 表示“允许访问”
- 因此，第一个 token 不能看到后面的位置，第二个 token 只能看到前两个位置，以此类推
- 这是标准的自回归掩码模式

- 批处理改变了这一情况，因为不同提示词通常长度不同
- 假设我们同时处理 `"2+2?"` 和稍长的提示词 `"3+3=6?"`
- 由于 PyTorch 张量必须是矩形形状，较短的行必须填充到与较长行相同的长度
- 这里使用左填充：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- 注意，我们在内部还保留一个额外的 `attn_mask`；它用于记录哪些位置是填充位置
- 在这个 `attn_mask` 中，`True` 表示已填充，`False` 表示未填充
- 我们使用额外的 `attn_mask`，来识别因果掩码中对应填充 token ID 的位置
- 对填充的键进行掩码、并将填充查询置零，是让批处理行为接近单样例执行的重要步骤

- 顺便一提，我们使用 `<|endoftext|>` token，但这并不重要，因为相应的 token 位置无论如何都会被忽略

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 第 3 章：批量评估 MATH-500

- 补充材料包含第 3 章评估方法的脚本，可以下载并像第 6 章那样使用：

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- 然后，在代码终端中执行下面的命令即可（如果你不用 uv，可将 `uv run` 替换为 `python`）：

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- bonus 材料还包含一个批量生成版本，它采用前面讨论的批处理方法
- 下载方式与之前类似，只需将 `evaluate_math500.py` 替换为 `evaluate_math500_batched.py`

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- 用法与非批量版本类似，但现在要额外提供 `--batch_size` 参数，指定 LLM 要并行处理多少个提示词和答案

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- 理想的批量大小取决于硬件的承受能力；批量大小为 64 时约使用 23.39 GB 内存（非批量脚本约使用 1.84 GB 内存）
- 本附录末尾会比较并讨论性能差异

&nbsp;
## E.5 第 4 章：批量自洽采样

- 实现第 4 章自洽采样的可选脚本 `self_consistency_math500_batched.py` 不会把不同提示词混合到一个经过填充的张量中
- 它会将同一个提示词重复 `num_samples` 次，并行采样多个延续结果，用于自洽投票
- 由于每一行都从相同的提示词长度开始，该脚本使用 `reasoning_from_scratch.qwen3` 中常规的 `Qwen3Model`，而不是 `reasoning_from_scratch.qwen3_batched`，因为长度相同就不需要填充

- 可以按如下方式下载脚本：

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- 下载非批量版本时，只需从上面的文件名中去掉 `"_batched"`
- 可以按如下方式运行脚本（非批量脚本的语法相同）：

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- 本附录末尾会介绍更多性能信息

&nbsp;
## E.6 第 6 章：批量 GRPO rollout

- 第 5 章的自我改进是一种顺序技术，本身无法从批处理中受益
- 可以并行运行多个输入的自我改进循环，但实现起来并不简单，因此不在补充材料中提供
- 我们改为继续讨论第 6 章 RLVR 的批量版本
- 第 6 章的不同 rollout 使用相同的提示词，因此不需要填充；和 E.5 节类似，代码使用 `reasoning_from_scratch.qwen3` 中常规的 `Qwen3Model` 类
- 相关脚本可以这样获取：

In [ ]:
# Non-batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# Batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# Batched version with GPU support
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- 在当前脚本中，`--batch_size` 控制每一步并行生成的 rollout 数量
- 这会提高吞吐量，但也会增加内存压力，因此实践中可能需要减小 `--num_rollouts` 或 `--max_new_tokens`
- 如果有多块 GPU，FSDP 版本遵循相同模式，并增加 `--num_gpus`
- 本附录末尾会继续讨论性能
- 截至本文撰写时，补充材料中还没有第 7 章脚本的批量版本，但后续会逐步加入；从概念上说，它们的工作方式会与第 6 章脚本类似

&nbsp;
## E.7 第 8 章：批量蒸馏

- 第 8 章回到了第 3 章那种考虑填充的方式，因为蒸馏样例的提示词和答案长度不同
- 可以按如下方式下载脚本和示例训练数据集：

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- 对于非批量版本，只需从文件名中去掉 `"_batched"`
- 可以按如下方式运行脚本：

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 单序列与批量生成

- 下表汇总了上述脚本的运行时间和内存占用：

| 行 | 脚本                                      | 批量大小 | 内存     | H100 总时间（分钟） | DGX Spark 总时间（分钟） |
|-----|-------------------------------------------|----------|----------|----------------------|---------------------------|
| 1   | evaluate_math500.py                       | -        | 1.8 GB   | 90.0                 | 174.7                     |
| 2   | evaluate_math500_batched.py               | 64       | 23.39 GB | 16.0                 | 108.4                     |
|     |                                           |          |          |                      |                           |
| 3   | self_consistency_math500.py               | -        | 1.79 GB  | 252.0                | 340.8                     |
| 4   | self_consistency_math500_batched.py       | 3        | 2.45 GB  | 129.0                | 243.3                     |
|     |                                           |          |          |                      |                           |
| 5   | rlvr_grpo_original_no_kl.py               | -        | 43.35 GB | 68.0                 | 63.7                      |
| 6   | rlvr_grpo_original_no_kl_batched.py       | 4        | 44.91 GB | 19.0                 | 23.1                      |
|     |                                           |          |          |                      |                           |
| 7   | distill.py                                | -        | 8.29 GB  | 10.9                 | 32.8                      |
| 8   | distill_batched.py                        | 4        | 8.34 GB  | 9.1                  | 28.2                      |